# Landing Zone

Objetivo de la fase:

1. Localizar los datasets originales.
2. Guardarlos como copias raw versionadas dentro de `landing_zone/raw`.
3. Devolver en el output del notebook el inventario de la ingesta.

Esta fase es la unica que persiste CSV.


In [1]:
import shutil
from datetime import UTC, datetime
from pathlib import Path

ROOT = Path.cwd()
LANDING_ZONE = ROOT / "landing_zone"
LANDING_RAW = LANDING_ZONE / "raw"
DATASETS = {
    "pics": {"filename": "opendatabcn_pics-csv.csv", "encoding": "utf-16"},
    "weather": {"filename": "dataset_prat_temps_2025-26.csv", "encoding": "utf-8-sig"},
    "hotels": {"filename": "opendatabcn_allotjament_hotels-csv.csv", "encoding": "utf-16"},
    "airbnb_listings": {"filename": "listings.csv", "encoding": "utf-8-sig"},
    "airbnb_neighbourhoods": {"filename": "neighbourhoods.csv", "encoding": "utf-8-sig"},
    "airbnb_reviews": {"filename": "reviews_airbnb.csv", "encoding": "utf-8-sig"},
    "airbnb_calendar": {"filename": "calendar.csv", "encoding": "utf-8-sig"},
    "income_2022": {"filename": "2022_renda_disponible_llars_per_persona.csv", "encoding": "utf-8-sig"},
    "hut_licenses": {"filename": "hut_comunicacio_opendata.csv", "encoding": "utf-8-sig"},
}


def ensure_phase_dirs() -> None:
    LANDING_RAW.mkdir(parents=True, exist_ok=True)


In [2]:
def utc_run_id() -> str:
    return datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")


def latest_landing_file(dataset: str) -> Path | None:
    dataset_dir = LANDING_RAW / dataset
    files = sorted(path for path in dataset_dir.glob("*") if path.is_file())
    return files[-1] if files else None


In [3]:
def run_collectors(run_id: str) -> dict[str, Path]:
    landing_paths: dict[str, Path] = {}
    for dataset, meta in DATASETS.items():
        source = ROOT / meta["filename"]
        target_dir = LANDING_RAW / dataset
        target_dir.mkdir(parents=True, exist_ok=True)
        if source.exists():
            target = target_dir / f"{run_id}_{source.name}"
            shutil.copy2(source, target)
        else:
            target = latest_landing_file(dataset)
            if target is None:
                raise FileNotFoundError(f"No se encuentra el CSV fuente ni una copia previa para {dataset}: {source}")
        landing_paths[dataset] = target
    return landing_paths


In [4]:
ensure_phase_dirs()
run_id = utc_run_id()
landing_paths = run_collectors(run_id)
{
    "run_id": run_id,
    "datasets": {name: str(path.relative_to(ROOT)) for name, path in landing_paths.items()},
}


{'run_id': '20260410T104218Z', 'datasets': {'pics': 'landing_zone/raw/pics/20260404T112053Z_opendatabcn_pics-csv.csv', 'weather': 'landing_zone/raw/weather/20260404T112053Z_dataset_prat_temps_2025-26.csv', 'hotels': 'landing_zone/raw/hotels/20260404T112053Z_opendatabcn_allotjament_hotels-csv.csv', 'airbnb_listings': 'landing_zone/raw/airbnb_listings/20260404T112053Z_listings.csv', 'airbnb_neighbourhoods': 'landing_zone/raw/airbnb_neighbourhoods/20260404T112053Z_neighbourhoods.csv', 'airbnb_reviews': 'landing_zone/raw/airbnb_reviews/20260404T112053Z_reviews_airbnb.csv', 'airbnb_calendar': 'landing_zone/raw/airbnb_calendar/20260404T112053Z_calendar.csv', 'income_2022': 'landing_zone/raw/income_2022/20260404T112053Z_2022_renda_disponible_llars_per_persona.csv', 'hut_licenses': 'landing_zone/raw/hut_licenses/20260404T112053Z_hut_comunicacio_opendata.csv'}}